# Analyze and Plot - NYC Taxi Data

This notebook computes metrics and generates visualizations from the processed taxi data.

In [ ]:
# Install necessary libraries (no requirements.txt)
import sys
import subprocess
import importlib

def _ensure(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pip_name])

_ensure("pandas", "pandas")
_ensure("matplotlib", "matplotlib")
_ensure("seaborn", "seaborn")

In [ ]:
# Import libraries
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set ggplot2-inspired style
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)
sns.set_palette("deep")

# Custom ggplot2-like settings
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#EBEBEB'
plt.rcParams['axes.edgecolor'] = 'white'
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['grid.color'] = 'white'
plt.rcParams['grid.linewidth'] = 1.2
plt.rcParams['xtick.color'] = '#4D4D4D'
plt.rcParams['ytick.color'] = '#4D4D4D'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']

# Define ggplot2-inspired color palette
ggplot_colors = ['#F8766D', '#7CAE00', '#00BFC4', '#C77CFF', '#FF6F00', '#00BA38', '#619CFF', '#F564E3']

In [ ]:
# Setup directories and read processed data
processed_dir = Path(os.getenv("TAXI_PROCESSED_DIR", "processed"))
plots_dir = Path(os.getenv("TAXI_PLOTS_DIR", "plots"))
metrics_dir = Path(os.getenv("TAXI_METRICS_DIR", "metrics"))
plots_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

clean = pd.read_csv(processed_dir / "trips_clean.csv")
agg = pd.read_csv(processed_dir / "aggregates_by_hour.csv")

print(f"Loaded {len(clean)} clean trips", flush=True)

In [ ]:
# Compute metrics
metrics = {
    "trips": int(len(clean)),
    "total_revenue": float(clean["total_amount"].sum()) if "total_amount" in clean.columns else None,
    "avg_total_amount": float(clean["total_amount"].mean()) if "total_amount" in clean.columns else None,
    "avg_distance_mi": float(clean["trip_distance"].mean()) if "trip_distance" in clean.columns else None,
    "avg_duration_min": float(clean["trip_duration_min"].mean()) if "trip_duration_min" in clean.columns else None,
}

if "payment_type" in clean.columns:
    counts = clean["payment_type"].value_counts(dropna=False).to_dict()
    metrics["payment_type_counts"] = {str(k): int(v) for k, v in counts.items()}

(metrics_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Wrote {metrics_dir / 'metrics.json'}", flush=True)

In [ ]:
# Plot 1: Trip distance histogram
if "trip_distance" in clean.columns:
    max_dist = float(os.getenv("TAXI_PLOT_MAX_DISTANCE", "15.0"))
    d = clean["trip_distance"].dropna()
    d = d[d <= max_dist]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(d, bins=30, color=ggplot_colors[0], alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title("Trip Distance Distribution", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Distance (miles)", fontsize=12, fontweight='500')
    ax.set_ylabel("Number of Trips", fontsize=12, fontweight='500')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "trip_distance_hist.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 2: Trip duration histogram
if "trip_duration_min" in clean.columns:
    max_dur = float(os.getenv("TAXI_PLOT_MAX_DURATION", "60.0"))
    x = clean["trip_duration_min"].dropna()
    x = x[x <= max_dur]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(x, bins=30, color=ggplot_colors[1], alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title("Trip Duration Distribution", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Duration (minutes)", fontsize=12, fontweight='500')
    ax.set_ylabel("Number of Trips", fontsize=12, fontweight='500')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "trip_duration_hist.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 3: Trips by pickup hour
if "pickup_hour" in agg.columns and "trips" in agg.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(agg["pickup_hour"], agg["trips"], color=ggplot_colors[2], alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title("Trips by Hour of Day", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Hour of Day", fontsize=12, fontweight='500')
    ax.set_ylabel("Number of Trips", fontsize=12, fontweight='500')
    ax.set_xticks(range(0, 24))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "trips_by_hour.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 10: Tip amount distribution (if available)
if "tip_amount" in clean.columns:
    max_tip = float(os.getenv("TAXI_PLOT_MAX_TIP", "20.0"))
    tips = clean["tip_amount"].dropna()
    tips = tips[tips <= max_tip]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(tips, bins=30, color=ggplot_colors[4], alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title("Tip Amount Distribution", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Tip Amount ($)", fontsize=12, fontweight='500')
    ax.set_ylabel("Number of Trips", fontsize=12, fontweight='500')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "tip_amount_hist.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 9: Payment type distribution (if available)
if "payment_type" in clean.columns:
    payment_counts = clean["payment_type"].value_counts()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(range(len(payment_counts)), payment_counts.values, 
                   color=ggplot_colors[:len(payment_counts)], alpha=0.8, 
                   edgecolor='white', linewidth=0.5)
    ax.set_title("Payment Type Distribution", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Payment Type", fontsize=12, fontweight='500')
    ax.set_ylabel("Number of Trips", fontsize=12, fontweight='500')
    ax.set_xticks(range(len(payment_counts)))
    ax.set_xticklabels(payment_counts.index)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "payment_type_distribution.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 8: Distance vs Fare scatter plot
if "trip_distance" in clean.columns and "total_amount" in clean.columns:
    # Sample data for better visualization if dataset is large
    sample_size = min(5000, len(clean))
    sample = clean.sample(n=sample_size, random_state=42) if len(clean) > sample_size else clean
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(sample["trip_distance"], sample["total_amount"], 
               alpha=0.4, s=30, color=ggplot_colors[0], edgecolors='none')
    
    # Add trend line
    valid_data = sample[["trip_distance", "total_amount"]].dropna()
    z = np.polyfit(valid_data["trip_distance"], valid_data["total_amount"], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(valid_data["trip_distance"].min(), valid_data["trip_distance"].max(), 100)
    ax.plot(x_trend, p(x_trend), color=ggplot_colors[1], linewidth=2.5, 
            label=f'y = {z[0]:.2f}x + {z[1]:.2f}', linestyle='--')
    
    ax.set_title("Trip Distance vs Total Fare", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Trip Distance (miles)", fontsize=12, fontweight='500')
    ax.set_ylabel("Total Fare ($)", fontsize=12, fontweight='500')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=True, fancybox=False, shadow=False, framealpha=0.9)
    
    out = plots_dir / "distance_vs_fare_scatter.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 7: Average trip metrics by hour
if all(col in agg.columns for col in ["pickup_hour", "avg_distance", "avg_duration_min"]):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Average distance by hour
    ax1.plot(agg["pickup_hour"], agg["avg_distance"], marker='o', linewidth=2.5, 
             markersize=7, color=ggplot_colors[6], markerfacecolor=ggplot_colors[6],
             markeredgewidth=0)
    ax1.set_title("Average Trip Distance by Hour", fontsize=16, fontweight='600', pad=20)
    ax1.set_xlabel("Hour of Day", fontsize=12, fontweight='500')
    ax1.set_ylabel("Average Distance (miles)", fontsize=12, fontweight='500')
    ax1.set_xticks(range(0, 24))
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    # Average duration by hour
    ax2.plot(agg["pickup_hour"], agg["avg_duration_min"], marker='o', linewidth=2.5, 
             markersize=7, color=ggplot_colors[7], markerfacecolor=ggplot_colors[7],
             markeredgewidth=0)
    ax2.set_title("Average Trip Duration by Hour", fontsize=16, fontweight='600', pad=20)
    ax2.set_xlabel("Hour of Day", fontsize=12, fontweight='500')
    ax2.set_ylabel("Average Duration (minutes)", fontsize=12, fontweight='500')
    ax2.set_xticks(range(0, 24))
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    
    out = plots_dir / "avg_metrics_by_hour.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 6: Revenue by hour
if "pickup_hour" in agg.columns and "total_revenue" in agg.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(agg["pickup_hour"], agg["total_revenue"], marker='o', linewidth=2.5, 
            markersize=8, color=ggplot_colors[5], markerfacecolor=ggplot_colors[5], 
            markeredgewidth=0)
    ax.fill_between(agg["pickup_hour"], agg["total_revenue"], alpha=0.3, color=ggplot_colors[5])
    ax.set_title("Total Revenue by Hour of Day", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Hour of Day", fontsize=12, fontweight='500')
    ax.set_ylabel("Total Revenue ($)", fontsize=12, fontweight='500')
    ax.set_xticks(range(0, 24))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "revenue_by_hour.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 5: Total fare distribution
if "total_amount" in clean.columns:
    max_fare = float(os.getenv("TAXI_PLOT_MAX_FARE", "100.0"))
    fares = clean["total_amount"].dropna()
    fares = fares[fares <= max_fare]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(fares, bins=30, color=ggplot_colors[4], alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title("Total Fare Distribution", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Total Amount ($)", fontsize=12, fontweight='500')
    ax.set_ylabel("Number of Trips", fontsize=12, fontweight='500')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "total_fare_hist.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)

In [ ]:
# Plot 4: Cost per mile distribution
if "cost_per_mile" in clean.columns:
    max_cpm = float(os.getenv("TAXI_PLOT_MAX_COST_PER_MILE", "20.0"))
    cpm = clean["cost_per_mile"].dropna()
    cpm = cpm[cpm <= max_cpm]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(cpm, bins=30, color=ggplot_colors[3], alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title("Cost per Mile Distribution", fontsize=16, fontweight='600', pad=20)
    ax.set_xlabel("Cost per Mile ($)", fontsize=12, fontweight='500')
    ax.set_ylabel("Number of Trips", fontsize=12, fontweight='500')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    out = plots_dir / "cost_per_mile_hist.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150, facecolor='white')
    plt.show()
    print(f"Wrote {out}", flush=True)